# Phase 5 (Legal Statute Identification, LSI) — Run #3 on the Phase-1 **Encoder-Decoder model 01** (bidirectional encoder), with **bottleneck adapters**

Config-driven like the other notebooks: edit **Cell 2** only. This is the **third** Phase-5 run: the **encoder** of the Phase-1 encoder-decoder, adapted with **adapter tuning** (Module 4.1 H: serial bottleneck adapters, Houlsby et al. 2019). Data, chunking, the label-attention head, the loss, thresholds, the submission writer and the run log are **identical to runs #1 and #2**; only the backbone and the adaptation method differ. `METHOD` also offers `"full"`, `"lorma"`, `"partial"` and `"head_only"`.

**Backbone.** Only the encoder is used (decoder, cross-attention and LM head are dropped): embeddings 16.38M + 6 bidirectional blocks 19.27M = **35.66M**. It was pretrained with masked-LM (15 %, 80/10/10) plus the seq2seq loss, so each token state sees **both sides natively**, no mask patch needed. Six layers instead of twelve: about half the cost per token of the decoders.

**Chunk length = the encoder's trained length.** The encoder was trained on 768-token inputs (`encoder_len`; RoPE table 1024). Positions 768–1023 were never seen in training, so `CHUNK_TOKENS = 768` (no RoPE extrapolation) and `MAX_CHUNKS = 5`: 5 × 767 = 3,835 tokens, close to the 4 × 1,023 = 4,092 of runs #1–#2 (≈ 95 % of all tokens read; the audit's coverage table lies between its 4×1024 and 4×512 rows). The comparison with the decoders stays fair up to that 6 % budget difference, which is worth one sentence in the report.

### The task in one paragraph
Each document (`{"id", "text", "labels"}`) has **one or more** of `L = 100` statute ids (0–99). Predict a **set** per document. Metric: **Macro-F1 over the 100 labels**, then micro-F1 and subset accuracy. Submission: `[{"id": …, "pred": [ints]}]`, unique valid ids, one object per test document.

### What the data audit (Step 0) found, and what it decided here
| Audit finding | Consequence in this notebook |
|---|---|
| 42,750 train / 10,181 dev / 13,019 test; **all 100 labels occur in train and dev**, rarest has 272 train docs, every label has ≥ 10 dev positives | Imbalance (43×) but no truly rare labels → per-label thresholds are *feasible*; whether they *help* is decided by a held-out check |
| **3.7 statutes per document** (max 31), no empty label sets | sigmoid per label (binary relevance), "at least one statute" rule is safe |
| Strong co-occurrence (lift up to 66, 2,036 pairs with ≥ 20 docs) | KD soft labels carry co-occurrence information; error analysis checks co-occurring pairs |
| Tokens/doc: median 657 (dev 935), p90 ≈ 4k, p99 ≈ 17–21k | `CHUNK_TOKENS = 1024`, `MAX_CHUNKS = 4` reads **95.6 %** of all tokens and **all of 88 %** of the documents; most documents are a single chunk |
| Explicit statute mentions in < 2 % of docs; **placeholder-like strings in 89 %** (≈ 5 per doc) | Statutes are **masked**: the model must infer them from facts. The placeholders still mark *where* a statute was cited → the `evidence` view and the attention-evidence analysis |
| TF-IDF + one-vs-rest logistic regression: **dev Macro-F1 0.397** (held-out tuned) | The bar every Phase-1 run is compared against |

### Model = Phase-1 backbone per chunk → **label-wise attention over all tokens of the document** → one logit per statute
1. Tokenise the document; cut it into chunks of `CHUNK_TOKENS` (each starts with `[BOS]`); keep at most `MAX_CHUNKS` (`DOC_VIEW` decides which, only for the 12 % of documents longer than the budget).
2. Encode each chunk with the (adapter-tuned) Phase-1 encoder, bidirectional by construction: `H` = token states, `d = 512`.
3. **Label-wise attention** (`AGG="label_attn"`, Vu et al. 2020, *LAAT*). This is the additive (Bahdanau) attention of Module 2.1 with **one query per statute**:
   `u_t = tanh(W h_t)`,  `α_{l,t} = softmax_t(q_lᵀ u_t)` over *all tokens of all chunks of the document*,  `c_l = Σ_t α_{l,t} h_t`,  `z_l = w_lᵀ c_l + b_l`.
   Why: a document cites ~4 statutes, usually in different passages. One pooled vector (`AGG="mean"` / `"attn"`) must summarise all of them at once; label-wise attention lets statute `l` read the passage that supports *it*. Cost: `W` (512×512) + `q`, `w` (100×512 each) ≈ 0.37M parameters.
4. Loss: **BCE with per-label `pos_weight_l = min(√((N−n_l)/n_l), cap)`** (the multi-label version of the Phase-3/4 sqrt-inverse class weights), or **ASL** (`LOSS="asl"`, see below).
5. **Prior-bias initialisation**: `b_l = log(π_l/(1−π_l))`, `π_l = n_l/N`. With `b = 0` every statute starts at `p = 0.5` although the average label is positive for only 3.7 % of documents; the first hundreds of steps would be spent pushing 96 % of the outputs down, and the rare labels get pushed to "never" before the features have learned anything. With the prior init, step 0 already predicts each statute at its base rate (Lin et al. 2017, focal-loss paper).

### Thresholds (where Macro-F1 is won or lost)
Macro-F1 = (1/L) Σ_l F1_l, and F1_l depends only on the decisions for label `l`, so per-label thresholds are `L` independent 1-D searches. The audit confirmed **θ\* ≈ F\*/2** for calibrated scores (Lipton et al. 2014; correlation 0.91 on 15 labels), so good thresholds are well below 0.5. With `pos_weight` the scores are deliberately *not* calibrated, which is why the threshold is tuned. Four strategies, each followed by **at least one statute per document** (top-1 if nothing clears its threshold):

| `strategy` | thresholds |
|---|---|
| `fixed` | 0.5 for every label |
| `global` | one θ tuned on dev |
| `per_label` | θ_l tuned on dev for each label |
| `shrunk` | θ_l = (n_l·θ̂_l + κ·θ_global)/(n_l + κ): the per-label estimate is pulled to the global one when label `l` has few dev positives `n_l` (a Bayesian-style shrinkage) |

The notebook picks the strategy with the best **2-fold held-out** score on dev (tune on one half, score the other, both ways). In-sample tuned numbers are optimistic. Report the *held-out* number and the untuned one.

### ASL (`LOSS="asl"`, Ridnik et al. 2021) in two lines
With `p = σ(z)`, `p_m = max(p − m, 0)`: `ℓ = −y (1−p)^{γ+} log p − (1−y) p_m^{γ−} log(1 − p_m)`. The focusing factor `p_m^{γ−}` down-weights easy negatives (a statute that is clearly absent), and the margin `m` *discards* very easy negatives completely (`p < m` ⇒ zero loss and zero gradient). Defaults `γ+ = 0, γ− = 4, m = 0.05`. In multi-label with 3.7 positives out of 100 labels, 96 % of the loss terms are negatives, most of them easy: that is the imbalance ASL targets. It is an alternative to `pos_weight`, not an addition (no `pos_weight` with ASL).

### Bottleneck adapters (Houlsby et al. 2019; Module 4.1 H)
Freeze every Phase-1 weight and insert a small trainable module **in series** inside each sub-layer:

`Adapter(y) = y + W_up · f(W_down · y)`,  `W_down ∈ R^{m×d}`, `W_up ∈ R^{d×m}`, `m ≪ d`, `f` = GELU.

It sits on the output of the sub-layer's last projection (`o_proj` of self-attention, `down_proj` of the SwiGLU FFN), **before** the residual addition, which is Houlsby's placement: `x ← x + Adapter(Sublayer(Norm(x)))`. Three properties matter:

* **Identity at initialisation.** `W_up = 0` ⇒ `Adapter(y) = y`, so the adapted encoder is *exactly* the Phase-1 encoder at step 0 (asserted at build time, like LoRMA's `B = 0`). At step 0 only `W_up` receives a gradient (`∂L/∂W_down` contains `W_upᵀ = 0`); after one step both do. Same mechanism as LoRA's `B = 0`.
* **The bottleneck is the capacity knob.** Per adapter: `2·d·m + m + d` parameters. With `d = 512`, `m = 64`: 66,112. Two adapters per block × 6 blocks = **0.79M** (2.2 % of the 35.66M encoder). Houlsby also trains the **layer-norm gains** (`train_norms=True`, +6.7k): renormalising the features the adapters now produce is cheap and helps.
* **Adapters vs LoRA / LoRMA.** LoRA adds a *linear* low-rank update `ΔW = BA` **in parallel** with a frozen weight, so it can be merged into `W` after training (zero inference overhead). An adapter is a **non-linear** function (`f` between the two projections) applied **in series** to the sub-layer's output: it can express things a low-rank linear update cannot, but it cannot be merged, and it adds two small sequential matmuls per sub-layer at inference. That latency cost is the standard argument for LoRA-style methods, and the run log records inference ms/doc so the report can check it against runs #1–#2.

`sites=("attn","ffn")` is Houlsby (2 per block); `("ffn",)` is the cheaper Pfeiffer-style placement (1 per block, half the parameters).

**KD note.** The teacher must be *larger* than the student; this encoder (35.66M) is below the 36.58M Dec-03 student, so `EXPORT_TEACHER=False`.

### What this notebook writes (`WORK_DIR/runs/<RUN_NAME>/`)
`submission/submission.zip` (verified like the evaluator would), `thresholds.json`, `run_log.json` (parameters, peak GPU memory, train time, inference ms/doc, dev metrics for every threshold strategy, per-label P/R/F1, F1 by label frequency and by document length, co-occurring-pair analysis, attention-evidence analysis), `dev_errors.json`, `dev_scores.npy` / `test_scores.npy` (float16 probabilities), `lsi_<RUN_NAME>.pt` (trainable weights), and (if `EXPORT_TEACHER`, off for this run) `teacher_logits_<RUN_NAME>.pt`.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy
from pathlib import Path
from dataclasses import dataclass, asdict, field, fields
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p5_encdec_adapter"
METHOD     = "adapter"        # "adapter" | "full" | "lorma" | "partial" | "head_only"
SMOKE_TEST = False            # True: ~150 steps on a small random subset, dev check only, NO submission written
ATTN_MODE  = "bidir"          # FIXED: the encoder is bidirectional by construction (kept as a constant for the shared cells)

# ---- Phase-1 checkpoint (written by caps_enc_dec_architecture_01.ipynb; must have encoder_causal=False) -------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # files are Phase_5_LSI/{Train/train.json, Train/dev.json, Test/test.json}
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_5_LSI/ (checked first)
FINAL_FIT  = False                                # True: train on train+dev, NO validation (after the dev run; EPOCHS = best epoch, THRESH_FILE set)
# The audit's regex for the statute placeholders (Cell 13 prints the strings it actually matches -> paste them to Claude if unsure)
PLACEHOLDER_RE = r"\[(?:MASK|STATUTE|SECTION|REDACTED)[^\]]*\]|<(?:mask|statute|section)[^>]*>|\b(?:XXX+|_{4,})"

# ---- Long-document representation -----------------------------------------------------------
CHUNK_TOKENS = 768           # tokens per chunk INCLUDING the leading [BOS] = the encoder's TRAINED length (RoPE table: 1024)
MAX_CHUNKS   = 5             # 5 x 767 = 3,835 tokens (runs #1-#2: 4 x 1,023 = 4,092) -> ~95% of all tokens read
DOC_VIEW     = "evidence"    # only matters for documents longer than the budget (12%):
                             # "head" | "tail" | "head_tail" | "spread" | "evidence" (chunk 0 + the chunks with most placeholders)
MAX_FULL_CHARS = 120_000     # longer texts (< 0.5%) are cut to their first 2/3 + last 1/3 of this many characters before tokenising
AGG          = "label_attn"  # "label_attn" (one attention query per statute) | "attn" (one shared query) | "mean"
ATT_DIM      = 512           # d_a of the attention projection W (d -> d_a)
N_PROC       = 4             # worker processes for tokenisation (1 = serial)

# ---- Training -----------------------------------------------------------------------------
EPOCHS       = 5             # adapters train ~0.8M weights and converge slower than full FT; the 6-layer encoder is ~2x faster per epoch
MICRO_TOKENS = 8192          # padded tokens per micro-batch (n_chunks x longest chunk); lower it on OOM
STEP_DOCS    = 32            # documents per optimiser step (gradient accumulation over micro-batches)
WARMUP_FRAC  = 0.06
GRAD_CLIP    = 1.0
GRAD_CKPT    = False         # True: recompute block activations in the backward pass (less memory, ~30% slower)
TIME_BUDGET_MIN = 480        # stop training after this many minutes (best epoch kept); leaves time for dev/test/export in a 12h session
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "adapter":   dict(lr=1e-3, head_lr=1e-3, wd=0.0, bottleneck=64, sites=("attn", "ffn"), train_norms=True),  # sites ("ffn",) = Pfeiffer
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "lorma":     dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, variant="plus"),   # variant: "plus" (LoRMA+) | "pi" (LoRMA_pi)
    "partial":   dict(lr=2e-4, head_lr=1e-3, wd=0.01, top_k=2, decay=0.8),            # top_k of the 6 blocks unfrozen, lr x decay per block downwards
    "head_only": dict(lr=0.0,  head_lr=2e-3, wd=0.0),
}
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")   # LoRMA
HEAD_DROPOUT    = 0.1
LOSS            = "bce"                           # "bce" (with POS_WEIGHT) | "asl"
POS_WEIGHT      = "sqrt_inv"                      # "sqrt_inv" | "none"   (bce only)
MAX_POS_WEIGHT  = 20.0                            # audit: sqrt-inverse ranges 1.6 ... 12.5, so the cap does not bind
ASL = dict(gamma_pos=0.0, gamma_neg=4.0, clip=0.05)
PRIOR_BIAS_INIT = True

# ---- Thresholds (tuned on dev; see the first markdown cell) -----------------------------------
THRESH_GRID      = [round(x, 3) for x in np.arange(0.02, 0.951, 0.01)]
THRESH_SHRINK_K  = 30.0                           # kappa of the "shrunk" strategy (in dev positives)
THRESH_MIN_GAIN  = 0.002                          # a finer strategy must beat a coarser one by this much (2-fold held-out) to be chosen
THRESH_FILE      = None                           # FINAL_FIT=True: path of the thresholds.json written by the dev run
TFIDF_BASELINE   = 0.397                          # audit: TF-IDF + LR, dev Macro-F1 (held-out tuned) -- for the printed comparison only

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 3000, 1000

# ---- Knowledge distillation (run #5 of the Phase-5 plan): this run is the planned TEACHER --------------------------------
EXPORT_TEACHER = False            # this encoder (35.66M) is SMALLER than the 36.58M Dec-03 student, so it cannot be its teacher

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE == "bidir" and LOSS in ("bce", "asl")
assert DOC_VIEW in ("head", "tail", "head_tail", "spread", "evidence") and AGG in ("label_attn", "attn", "mean")
assert 16 <= CHUNK_TOKENS <= 1024 and MAX_CHUNKS >= 1, "chunks longer than 1024 exceed the Phase-1 RoPE table"
assert not (FINAL_FIT and SMOKE_TEST)
BODY = CHUNK_TOKENS - 1                           # tokens of text per chunk (the first position is [BOS])
SEQ_LEN = CHUNK_TOKENS                            # (used by the checkpoint-loading cell)
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | encoder-only (bidirectional) | view={DOC_VIEW} {MAX_CHUNKS}x{CHUNK_TOKENS} | agg={AGG} | loss={LOSS} | "
      f"micro={MICRO_TOKENS} tok, step={STEP_DOCS} docs | epochs={EPOCHS} | final_fit={FINAL_FIT} | smoke={SMOKE_TEST}")

RUN p5_encdec_adapter | METHOD=adapter | encoder-only (bidirectional) | view=evidence 5x768 | agg=label_attn | loss=bce | micro=8192 tok, step=32 docs | epochs=5 | final_fit=False | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_enc_dec_architecture_01.ipynb` (Cell 3)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# ===================== Cell 3: model =====================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8 / 3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024          # RoPE table size; must cover encoder_len and decoder_len
    encoder_len: int = 768
    decoder_len: int = 256
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"           # rope | learned
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"         # swiglu | gelu
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True
    encoder_causal: bool = False     # False = bidirectional encoder
    mlm_head: bool = True            # BERT-style MLM head on the encoder (auxiliary objective)
    arch_name: str = "encdec_bidir_mlm_v2"

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        assert self.max_seq_len >= max(self.encoder_len, self.decoder_len)
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return ((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype) * self.g + self.b

def build_norm(cfg):
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        T = q.shape[2]
        if position_ids is None:
            cos = self.cos[:T][None, None]; sin = self.sin[:T][None, None]
        else:                                    # position_ids: [B, T] or [1, T]
            cos = self.cos[position_ids].unsqueeze(1); sin = self.sin[position_ids].unsqueeze(1)
        dt = q.dtype
        q = self._rotate(q.float(), cos, sin); k = self._rotate(k.float(), cos, sin)
        return q.to(dt), k.to(dt)


class Attention(nn.Module):
    # Self-attention (optionally causal, with KV cache) or cross-attention (context / precomputed static_kv).
    def __init__(self, cfg, causal):
        super().__init__()
        self.cfg, self.causal = cfg, causal
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, nh):
        B, T, _ = x.shape
        return x.view(B, T, nh, self.head_dim).transpose(1, 2)

    def project_kv(self, src):
        return self._split(self.k_proj(src), self.n_kv_head), self._split(self.v_proj(src), self.n_kv_head)

    def _attend(self, q, k, v, causal, key_mask):
        T, S = q.size(2), k.size(2)
        dp = self.cfg.attn_dropout if self.training else 0.0
        mask, sdpa_causal = None, False
        if causal and T > 1:
            if T == S and key_mask is None:
                sdpa_causal = True
            else:   # general (offset) causal mask: query i sees keys <= i + (S - T)
                mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril(diagonal=S - T).view(1, 1, T, S)
        if key_mask is not None:                 # key_mask: [B, S], True/1 = real token
            km = key_mask.to(torch.bool).view(key_mask.size(0), 1, 1, S)
            mask = km if mask is None else (mask & km)
        if self.cfg.use_sdpa:
            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, is_causal=sdpa_causal)
        scores = (q.float() @ k.float().transpose(-2, -1)) * self.scale
        if sdpa_causal:
            mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril().view(1, 1, T, S)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        p = F.dropout(F.softmax(scores, dim=-1), dp, self.training).to(q.dtype)
        return p @ v

    def forward(self, x, rope=None, position_ids=None, key_mask=None, kv_cache=None,
                context=None, static_kv=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        is_cross = (context is not None) or (static_kv is not None)
        if static_kv is not None:
            k, v = static_kv
        elif context is not None:
            k, v = self.project_kv(context)
        else:
            k, v = self.project_kv(x)
            if rope is not None:
                q, k = rope(q, k, position_ids=position_ids)
            if kv_cache is not None:
                k = torch.cat([kv_cache[0], k], dim=2)
                v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = None if is_cross else (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        out = self._attend(q, k, v, causal=(self.causal and not is_cross), key_mask=key_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))

def build_mlp(cfg):
    return SwiGLU(cfg) if cfg.mlp_type == "swiglu" else GELUMLP(cfg)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=cfg.encoder_causal)
        self.norm2, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, rope=None, key_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, key_mask=key_mask)
        x = x + h
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=True)
        self.norm2, self.cross_attn = build_norm(cfg), Attention(cfg, causal=False)
        self.norm3, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None, rope=None,
                position_ids=None, kv_cache=None):
        h, new_self = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, kv_cache=kv_cache)
        x = x + h
        if cross_kv is not None:
            h, _ = self.cross_attn(self.norm2(x), static_kv=cross_kv, key_mask=enc_mask)
        else:
            h, _ = self.cross_attn(self.norm2(x), context=enc_out, key_mask=enc_mask)
        x = x + h
        return x + self.mlp(self.norm3(x)), new_self


class Encoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)
    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, key_mask=attention_mask, position_ids=position_ids)
        return self.norm_f(x)

class Decoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def precompute_cross_kv(self, enc_out):
        # Cross-attention K/V depend only on the encoder output: compute once per encoder pass.
        return [blk.cross_attn.project_kv(enc_out) for blk in self.blocks]

    def forward(self, input_ids, enc_out=None, enc_mask=None, position_ids=None,
                kv_caches=None, cross_kvs=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, enc_out=enc_out,
                        cross_kv=cross_kvs[i] if cross_kvs is not None else None,
                        enc_mask=enc_mask, rope=self.rope, position_ids=position_ids,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class Seq2SeqLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        # BERT-style transform before the (tied) output projection for MLM.
        self.mlm_transform = (nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=True), nn.GELU(),
                                            build_norm(cfg)) if cfg.mlm_head else None)

    def encode(self, enc_ids, enc_mask=None):
        return self.encoder(enc_ids, attention_mask=enc_mask)

    def forward(self, enc_ids, dec_ids, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        h, _ = self.decoder(dec_ids, enc_out=enc_out, enc_mask=enc_mask)
        return self.lm_head(h)

    def mlm_logits_at(self, enc_hidden, select):
        # Project only the masked positions -> [N, V] (avoids a full [B, T, V] tensor).
        return self.lm_head(self.mlm_transform(enc_hidden[select]))


def init_weights(model, cfg):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    # GPT-2-style residual scaling, per stack: encoder blocks add 2 residual branches, decoder blocks add 3.
    s_enc = 1.0 / math.sqrt(2 * cfg.n_encoder_layers)
    s_dec = 1.0 / math.sqrt(3 * cfg.n_decoder_layers)
    with torch.no_grad():
        for blk in model.encoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_enc); blk.mlp.down_proj.weight.mul_(s_enc)
        for blk in model.decoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_dec); blk.cross_attn.o_proj.weight.mul_(s_dec)
            blk.mlp.down_proj.weight.mul_(s_dec)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())   # .parameters() de-duplicates tied weights
    emb = model.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# ---- Checkpoint helpers (with the causal/bidirectional guard) ----
def cfg_from_dict(d):
    d = dict(d)
    if "encoder_causal" not in d:
        raise ValueError("Checkpoint config has no 'encoder_causal' key. This is probably an old causal-encoder "
                         "checkpoint. Refusing to guess: the flag is a mask, not a weight, so a wrong guess would "
                         "load silently and run the encoder in a mode it was never trained in. Add "
                         "ckpt['config']['encoder_causal'] = True/False manually if you know which run it is.")
    d.setdefault("mlm_head", False)          # pre-v2 checkpoints have no MLM head
    d.setdefault("arch_name", "legacy")
    known = {f.name for f in fields(ModelConfig)}
    dropped = sorted(set(d) - known)
    if dropped:
        print("cfg_from_dict: ignoring unknown keys", dropped)
    return ModelConfig(**{k: v for k, v in d.items() if k in known})

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_model_from_ckpt(path, expect_encoder_causal=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = cfg_from_dict(ck["config"])
    if expect_encoder_causal is not None and cfg.encoder_causal != expect_encoder_causal:
        raise ValueError(f"{path}: encoder_causal={cfg.encoder_causal}, expected {expect_encoder_causal}")
    model = Seq2SeqLM(cfg)
    model.load_state_dict(ck["model"], strict=True)
    st = ck.get("state", {})
    print(f"Loaded {path} | arch={cfg.arch_name} | encoder_causal={cfg.encoder_causal} | "
          f"enc_len={cfg.encoder_len} dec_len={cfg.decoder_len} | step={st.get('step', ck.get('step'))} | "
          f"val_loss={st.get('val_loss', ck.get('val_loss'))}")
    return model.to(device).eval(), cfg, ck

_c = ModelConfig(vocab_size=VOCAB_SIZE)
_pc = count_params(Seq2SeqLM(_c))
print(f"Default config params: total={_pc['total']/1e6:.2f}M (embedding {_pc['embedding']/1e6:.2f}M)")
del _c

Default config params: total=61.49M (embedding 16.38M)


## Load the Phase-1 checkpoint

In [6]:
# Cell 12 — Load the Phase-1 encoder-decoder checkpoint and sanity-check its ENCODER
lm, lm_cfg, _ck = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)       # refuses a causal-encoder checkpoint
del _ck
pc = count_params(lm)
n_enc = sum(p.numel() for p in lm.encoder.parameters())                              # embeddings (shared) + encoder blocks + final norm
print(f"Phase-1 model | arch={lm_cfg.arch_name} | enc layers={lm_cfg.n_encoder_layers} dec layers={lm_cfg.n_decoder_layers} "
      f"d_model={lm_cfg.d_model} pos={lm_cfg.pos_type} | total {pc['total']/1e6:.2f}M | ENCODER-ONLY (used here) {n_enc/1e6:.2f}M")
assert SEQ_LEN <= lm_cfg.max_seq_len, "window longer than the RoPE table of this checkpoint"
if SEQ_LEN > lm_cfg.encoder_len:
    print(f"WARNING: sequence length {SEQ_LEN} > the encoder's trained length {lm_cfg.encoder_len} (RoPE extrapolation, expect some degradation)")

# Sanity: if the weights did not load properly, masked-token prediction would sit near ln(V) ~ 10.4.
MASK_ID = getattr(tokenizer, "special_to_id", {}).get("<mask>", 6)
if lm_cfg.mlm_head:
    _sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
    _x = torch.tensor([[BOS_ID] + tokenizer.encode(_sent)], device=device)
    _pos = [p for p in (4, 9, 13) if p < _x.size(1) - 1]
    _xm = _x.clone(); _xm[0, _pos] = MASK_ID
    _sel = torch.zeros_like(_xm, dtype=torch.bool); _sel[0, _pos] = True
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        _h = lm.encode(_xm, torch.ones_like(_xm))
        _lg = lm.mlm_logits_at(_h, _sel)
    _l = F.cross_entropy(_lg.float(), _x[0, _pos]).item()
    print(f"masked-LM loss on a legal sentence (3 masked tokens): {_l:.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
    assert _l < 9.0, "checkpoint looks untrained / mis-loaded"
else:
    print("this checkpoint has no MLM head; skipping the masked-LM sanity check")

Loaded /kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt | arch=encdec_bidir_mlm_v2 | encoder_causal=False | enc_len=768 dec_len=256 | step=15488 | val_loss=1.9592824522405863
Phase-1 model | arch=encdec_bidir_mlm_v2 | enc layers=6 dec layers=6 d_model=512 pos=rope | total 61.49M | ENCODER-ONLY (used here) 35.66M
masked-LM loss on a legal sentence (3 masked tokens): 0.117  (ln V = 10.37)


## Phase-5 data: documents, statute sets, placeholders
`{"id", "text", "labels"}` (test: no `labels`). The loader accepts the same alternative layouts as in Step 0 and **fails loudly** on a non-integer label. The label list comes from train ∪ dev (audit: ids 0–99, all present in both). The cell also prints the **placeholder strings** that `PLACEHOLDER_RE` matches: they should look like a mask token (e.g. `<SECTION>` / `[STATUTE]`), not like a line of underscores. If the top strings look wrong, paste them to Claude.

In [7]:
# Cell 13 — Phase-5 data: train / dev / test documents, label space, placeholder strings
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

P5 = "Phase_5_LSI"

def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download Phase_5_LSI and set LOCAL_DATA_DIR.") from ex

def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default

LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "y", "targets")
def _to_int_list(lab, split, rid):
    if lab is None: raise ValueError(f"{split} doc {rid}: no label field (looked for {LABEL_KEYS})")
    if isinstance(lab, (int, np.integer)): lab = [lab]
    if isinstance(lab, str): lab = [t for t in re.split(r"[\s,;\[\]]+", lab) if t]
    out = []
    for v in lab:
        try: out.append(int(v))
        except Exception: raise ValueError(f"{split} doc {rid}: label {v!r} is not an integer -> paste this message to Claude")
    return sorted(set(out))

def parse_docs(raw, split, has_labels):
    """-> list of {uid, id, text, labels|None, nchar}.  uid = '<split>::<position in the file>' (stable, used by the caches and the KD export)."""
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"text": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ("id", "case_id", "doc_id", "document_id"), i)
        text = _first(e, ("text", "case_text", "document", "facts", "content"))
        if isinstance(text, list): text = "\n".join(str(t) for t in text)
        if not isinstance(text, str): raise ValueError(f"{split} doc {rid}: no text field. Keys: {list(e)}")
        labels = _to_int_list(_first(e, LABEL_KEYS), split, rid) if has_labels else None
        docs.append(dict(uid=f"{split}::{i}", id=rid, text=text, labels=labels, nchar=len(text)))
    return docs

def load_split(rel, split, has_labels):
    raw = json.loads(fetch(rel).read_text(encoding="utf-8"))
    first = raw[0] if isinstance(raw, list) and raw else (next(iter(raw.values())) if isinstance(raw, dict) and raw else None)
    print(f"  {rel}: {type(raw).__name__} of {len(raw)} | first item keys: {list(first) if isinstance(first, dict) else type(first).__name__}")
    return parse_docs(raw, split, has_labels)

TRAIN_ALL = load_split(f"{P5}/Train/train.json", "train", True)
DEV_ALL   = load_split(f"{P5}/Train/dev.json",   "dev",   True)
TEST_ALL  = load_split(f"{P5}/Test/test.json",   "test",  False)

LABELS = sorted({l for d in TRAIN_ALL + DEV_ALL for l in d["labels"]})
L = len(LABELS); LAB2IDX = {l: j for j, l in enumerate(LABELS)}
def to_multihot(docs):
    Y = np.zeros((len(docs), L), dtype=np.float32)
    for i, d in enumerate(docs):
        for l in d["labels"]: Y[i, LAB2IDX[l]] = 1.0
    return Y
for d in TRAIN_ALL + DEV_ALL: d["y"] = to_multihot([d])[0]
for name, ds in (("train", TRAIN_ALL), ("dev", DEV_ALL), ("test", TEST_ALL)):
    nc = np.asarray([d["nchar"] for d in ds]); msg = f"{name:5s}: {len(ds):6d} docs | chars/doc median {np.median(nc):7.0f} p90 {np.percentile(nc, 90):7.0f} max {nc.max():7d}"
    if ds[0]["labels"] is not None: msg += f" | statutes/doc {np.mean([len(d['labels']) for d in ds]):.2f}"
    print(msg)
print(f"label space: L = {L} | ids {LABELS[0]}..{LABELS[-1]} | contiguous {LABELS == list(range(LABELS[0], LABELS[-1] + 1))}")
_ti = [str(d["id"]) for d in TEST_ALL]
assert len(set(_ti)) == len(_ti), "test ids are not unique -> paste this to Claude (the submission needs one prediction per id)"

PH_RX = re.compile(PLACEHOLDER_RE, re.I)
_ph = Counter()
for d in random.Random(SEED).sample(TRAIN_ALL, min(3000, len(TRAIN_ALL))):
    _ph.update(m.group(0) for m in PH_RX.finditer(d["text"]))
print("most frequent placeholder strings:", _ph.most_common(12))
_ex = next((d for d in TRAIN_ALL[:2000] if PH_RX.search(d["text"])), None)
if _ex is not None:
    m = PH_RX.search(_ex["text"]); a = max(0, m.start() - 150)
    print("context of one match:", repr(_ex["text"][a:m.end() + 150]))

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

  Phase_5_LSI/Train/train.json: list of 42750 | first item keys: ['id', 'text', 'labels']


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

  Phase_5_LSI/Train/dev.json: list of 10181 | first item keys: ['id', 'text', 'labels']


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

  Phase_5_LSI/Test/test.json: list of 13019 | first item keys: ['id', 'text']
train:  42750 docs | chars/doc median    2717 p90   16789 max  535964 | statutes/doc 3.70
dev  :  10181 docs | chars/doc median    3418 p90   17900 max  442434 | statutes/doc 3.92
test :  13019 docs | chars/doc median    3368 p90   17277 max  820502
label space: L = 100 | ids 0..99 | contiguous True
most frequent placeholder strings: [('<SECTION>', 15496), ('xxx', 60), ('xxxx', 16), ('xxxxx', 11), ('XXX', 9), ('XXXXXX', 7), ('XXXX', 2), ('xxxxxx', 2), ('xxxxxxxxx', 2), ('___________________', 1), ('xxxxxxx', 1)]
context of one match: '(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commer'


## Chunks: selection (views), tokenisation, cache, length-bucketed micro-batches
Every document is tokenised in full (with character offsets, so the placeholder positions are known in token space), cut into chunks of `BODY = CHUNK_TOKENS − 1` tokens from the start, and `[BOS]` is prepended to each chunk. If a document has more than `MAX_CHUNKS` chunks, `DOC_VIEW` picks which ones:

| `DOC_VIEW` | kept chunks (only documents with more than `MAX_CHUNKS` chunks) |
|---|---|
| `evidence` (default) | chunk 0 (parties / facts) **+ the `MAX_CHUNKS−1` chunks that contain the most placeholders**, in document order. A placeholder is where a statute was cited, so these chunks are the evidence (the PDF's "evidence-based context selection") |
| `head` / `tail` | the first / last chunks |
| `head_tail` | half from the start, half from the end |
| `spread` | evenly spaced over the document |

Micro-batches hold documents of similar length (sorted inside shuffled pools), and are cut by a **token budget** (`MICRO_TOKENS` = number of chunks × longest chunk), so a batch of short documents carries many documents and a batch of long ones few. Gradients are accumulated until `STEP_DOCS` documents.

In [8]:
# Cell 15 — Chunk selection, tokenisation (cached, multi-process), batching
def _cut_text(t):
    if len(t) <= MAX_FULL_CHARS: return t
    h = 2 * MAX_FULL_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_FULL_CHARS - h):]

def _tok_with_placeholders(text):
    """-> (ids list, bool array: token overlaps a placeholder string)."""
    text = _cut_text(text)
    ids, offs = tokenizer.encode(text, return_offsets=True)
    ids = list(ids); isph = np.zeros(len(ids), dtype=bool)
    spans = [(m.start(), m.end()) for m in PH_RX.finditer(text)]
    if spans and ids:
        starts = np.asarray([a for a, _ in offs]); ends = np.asarray([b for _, b in offs])
        for a, b in spans:
            lo = np.searchsorted(ends, a, side="right"); hi = np.searchsorted(starts, b, side="left")
            isph[lo:hi] = True
    return ids, isph

def _select(n_chunks, ph_counts):
    K = MAX_CHUNKS
    if n_chunks <= K: return list(range(n_chunks))
    if DOC_VIEW == "head": return list(range(K))
    if DOC_VIEW == "tail": return list(range(n_chunks - K, n_chunks))
    if DOC_VIEW == "head_tail":
        Kh = (K + 1) // 2; return list(range(Kh)) + list(range(n_chunks - (K - Kh), n_chunks))
    if DOC_VIEW == "spread":
        return sorted(set(np.linspace(0, n_chunks - 1, K).round().astype(int).tolist()))
    rest = sorted(range(1, n_chunks), key=lambda j: (-ph_counts[j], j))[:K - 1]          # "evidence"
    return sorted([0] + rest)

_TXT = None
def _work(i):
    ids, isph = _tok_with_placeholders(_TXT[i])
    if not ids: ids, isph = [EOS_ID], np.zeros(1, dtype=bool)
    starts = list(range(0, len(ids), BODY))
    ph_counts = [int(isph[s:s + BODY].sum()) for s in starts]
    keep = _select(len(starts), ph_counts)
    chunks = [np.asarray([BOS_ID] + ids[starts[j]:starts[j] + BODY], dtype=np.int32) for j in keep]
    phpos = [np.flatnonzero(isph[starts[j]:starts[j] + BODY]).astype(np.int16) + 1 for j in keep]     # +1: position 0 is [BOS]
    return chunks, phpos, len(ids), len(starts)

def tokenize_docs(docs, cache_name):
    """Adds d['chunks'] (int32 arrays, each starting with [BOS]), d['phpos'] (placeholder positions per chunk), d['ntok'], d['nchunks_all']."""
    global _TXT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"p5_{cache_name}_v{VOCAB_SIZE}_{DOC_VIEW}_c{CHUNK_TOKENS}_k{MAX_CHUNKS}_m{MAX_FULL_CHARS}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None,))[0] != d["nchar"]]
    if todo:
        _TXT = [d["text"] for d in docs]
        order = sorted(todo, key=lambda i: -docs[i]["nchar"])                     # long ones first: better load balance
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, order, chunksize=16), total=len(order), desc=f"tokenise {cache_name}", leave=False))
            except Exception as ex:
                print("parallel tokenisation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(order, desc=f"tokenise {cache_name}", leave=False)]
        for i, r in zip(order, res): cache[docs[i]["uid"]] = (docs[i]["nchar"],) + tuple(r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs:
        _, d["chunks"], d["phpos"], d["ntok"], d["nchunks_all"] = cache[d["uid"]]
    return docs

def chunk_stats(docs, title):
    nch = np.asarray([len(d["chunks"]) for d in docs]); ntok = np.asarray([d["ntok"] for d in docs])
    read = np.asarray([sum(len(c) - 1 for c in d["chunks"]) for d in docs])
    nph = np.asarray([sum(len(p) for p in d["phpos"]) for d in docs])
    print(f"{title}: {len(docs)} docs | tokens/doc median {np.median(ntok):.0f} p90 {np.percentile(ntok, 90):.0f} | chunks/doc mean {nch.mean():.2f} "
          f"(1 chunk: {100 * (nch == 1).mean():.0f}%, truncated by the view: {100 * np.mean([d['nchunks_all'] > MAX_CHUNKS for d in docs]):.1f}%) | "
          f"share of tokens read {100 * read.sum() / max(ntok.sum(), 1):.1f}% | docs with a placeholder token in the kept chunks {100 * (nph > 0).mean():.0f}%")

def make_microbatches(docs, idxs, shuffle, seed=0):
    """Group documents of similar length; cut by MICRO_TOKENS (n_chunks x longest chunk). Returns a list of index lists."""
    idxs = list(idxs)
    rng = random.Random(seed)
    if shuffle: rng.shuffle(idxs)
    pool = max(1, 50 * max(1, MICRO_TOKENS // CHUNK_TOKENS))
    batches = []
    for p in range(0, len(idxs), pool):
        part = sorted(idxs[p:p + pool], key=lambda i: (len(docs[i]["chunks"]), max(len(c) for c in docs[i]["chunks"])))
        cur, n_ch, longest = [], 0, 0
        for i in part:
            c = docs[i]["chunks"]; ln = max(len(x) for x in c)
            if cur and (n_ch + len(c)) * max(longest, ln) > MICRO_TOKENS:
                batches.append(cur); cur, n_ch, longest = [], 0, 0
            cur.append(i); n_ch += len(c); longest = max(longest, ln)
        if cur: batches.append(cur)
    if shuffle: rng.shuffle(batches)
    return batches

def build_batch(docs, idxs, with_labels=True):
    """-> ids [N, T], mask [N, T], owner [N] (doc position in the batch), slot [N] (chunk position in its doc), y [B, L] or None."""
    seqs, owner, slot = [], [], []
    for b, di in enumerate(idxs):
        for j, c in enumerate(docs[di]["chunks"]):
            seqs.append(c); owner.append(b); slot.append(j)
    T = max(len(s) for s in seqs)
    ids = np.full((len(seqs), T), PAD_ID, dtype=np.int64); mask = np.zeros((len(seqs), T), dtype=np.int64)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1
    y = np.stack([docs[di]["y"] for di in idxs]) if with_labels else None
    return ids, mask, np.asarray(owner), np.asarray(slot), y

## Metrics and thresholds (pure numpy)
`scores(Y, P)` gives per-label P/R/F1, Macro-F1 (over the labels with at least one gold document in the evaluated set, = all 100 on dev), micro-F1 and subset accuracy. The threshold strategies and the 2-fold selection are described in the first markdown cell.

In [9]:
# Cell 16 — Multi-label metrics, threshold strategies, held-out selection
def scores(Y, P):
    Y = Y.astype(bool); P = P.astype(bool)
    tp = (Y & P).sum(0).astype(float); fp = (~Y & P).sum(0).astype(float); fn = (Y & ~P).sum(0).astype(float)
    sup = Y.sum(0); npred = P.sum(0)
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros_like(tp), where=(2 * tp + fp + fn) > 0)
    m = sup > 0
    micro = 2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)
    return dict(macro_f1=float(f1[m].mean()), macro_p=float(prec[m].mean()), macro_r=float(rec[m].mean()), micro_f1=float(micro),
                subset_acc=float((Y == P).all(1).mean()), f1=f1, p=prec, r=rec, support=sup, n_pred=npred,
                card_pred=float(P.sum(1).mean()), card_gold=float(Y.sum(1).mean()))

def decide(S, th):
    """Probabilities S [n, L] + thresholds (scalar or [L]) -> decisions, with at least one statute per document."""
    P = S >= np.asarray(th)
    empty = ~P.any(1)
    if empty.any(): P[np.where(empty)[0], S[empty].argmax(1)] = True
    return P

def _best_global(S, Y):
    return max(THRESH_GRID, key=lambda t: (scores(Y, decide(S, t))["macro_f1"], -abs(t - 0.5)))

def _best_per_label(S, Y):
    """Exact: F1_l depends only on theta_l. Returns (theta_hat [L], n_pos [L])."""
    Yb = Y.astype(bool); th = np.full(S.shape[1], 0.5); npos = Yb.sum(0)
    for j in range(S.shape[1]):
        y = Yb[:, j]
        if y.sum() == 0: continue
        best_f, best_t = -1.0, 0.5
        for t in THRESH_GRID:
            p = S[:, j] >= t; tp = (p & y).sum(); f = 2 * tp / max(p.sum() + y.sum(), 1)
            if f > best_f + 1e-12 or (abs(f - best_f) <= 1e-12 and abs(t - 0.5) < abs(best_t - 0.5)): best_f, best_t = f, t
        th[j] = best_t
    return th, npos

def fit_thresholds(S, Y, strategy):
    if strategy == "fixed": return np.full(S.shape[1], 0.5)
    tg = _best_global(S, Y)
    if strategy == "global": return np.full(S.shape[1], tg)
    th, npos = _best_per_label(S, Y)
    th = np.where(npos > 0, th, tg)
    if strategy == "per_label": return th
    return (npos * th + THRESH_SHRINK_K * tg) / (npos + THRESH_SHRINK_K)            # "shrunk"

STRATEGIES = ("fixed", "global", "per_label", "shrunk")
def heldout(S, Y, strategy, seed=SEED):
    """2-fold on the given set: fit on one half, score the other, both ways; mean Macro-F1."""
    perm = np.random.RandomState(seed).permutation(len(S)); h = (perm[: len(S) // 2], perm[len(S) // 2:])
    out = []
    for a, b in (h, h[::-1]):
        out.append(scores(Y[b], decide(S[b], fit_thresholds(S[a], Y[a], strategy)))["macro_f1"])
    return float(np.mean(out))

def choose_strategy(S, Y):
    """Coarse -> fine; a finer strategy must beat the best coarser one by THRESH_MIN_GAIN (held-out)."""
    ho = {s: heldout(S, Y, s) for s in STRATEGIES}
    best = "fixed"
    for s in ("global", "per_label", "shrunk"):
        if ho[s] >= ho[best] + THRESH_MIN_GAIN: best = s
    return best, ho

def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -40, 40)))

## Model, adaptation methods, loss
The LoRMA layer is the Phase-3/4 code, unchanged; no attention patch is needed (the encoder is bidirectional). New: the **bottleneck adapter**, the **multi-label document head** (label-wise attention over all tokens of the document), the **prior-bias initialisation**, the multi-label losses, and the `partial` method (Module 4.1 A: freeze the lower layers, update the upper ones; here with a learning rate that decays by `decay` per block going down, so the blocks closest to the head move most).

In [10]:
# Cell 17 — Bidirectional attention, the multi-label document model, the adaptation methods, the losses

print("attention: the Phase-1 encoder is bidirectional by construction (key-padding mask only)")


# ---- (b) Multi-label document model ------------------------------------------------------------------------------------------
class DocModelML(nn.Module):
    """Chunks -> Phase-1 backbone -> token states placed on a [B, K*T] document grid -> pooling (AGG) -> one logit per statute."""
    def __init__(self, backbone, d_model, n_labels, n_slots, agg, att_dim, dropout, prior=None):
        super().__init__()
        self.backbone, self.agg, self.K, self.L = backbone, agg, n_slots, n_labels
        self.drop = nn.Dropout(dropout)
        if agg in ("label_attn", "attn"):
            self.att_w = nn.Linear(d_model, att_dim)
            self.att_q = nn.Parameter(torch.randn(n_labels if agg == "label_attn" else 1, att_dim) * 0.02)
        if agg == "label_attn":                                                      # per-label output vector w_l and bias b_l
            self.out_w = nn.Parameter(torch.randn(n_labels, d_model) * 0.02)
            self.out_b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.head = nn.Linear(d_model, n_labels)
            nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)
        if prior is not None:                                                        # b_l = log(pi_l / (1 - pi_l))
            b = torch.log(torch.as_tensor(prior) / (1 - torch.as_tensor(prior))).float()
            with torch.no_grad():
                (self.out_b if agg == "label_attn" else self.head.bias).copy_(b)

    def encode(self, ids, mask):
        """Same computation as the Phase-1 Encoder.forward (bidirectional, key-padding mask), with optional activation checkpointing."""
        enc = self.backbone; N, T = ids.shape                                       # self.backbone is the Phase-1 Encoder here
        x = enc.tok_emb(ids); pos = None
        if enc.pos_emb is not None:
            pos = torch.arange(T, device=ids.device)[None]; x = x + enc.pos_emb(pos)
        x = enc.emb_drop(x)
        for blk in enc.blocks:
            if self.training and GRAD_CKPT and torch.is_grad_enabled():
                x = checkpoint(lambda t, blk=blk: blk(t, rope=enc.rope, key_mask=mask, position_ids=pos), x, use_reentrant=False)
            else:
                x = blk(x, rope=enc.rope, key_mask=mask, position_ids=pos)
        return enc.norm_f(x)

    def forward(self, ids, mask, owner, slot, B, return_attn=False):
        h = self.encode(ids, mask).float()                                          # [N, T, d]
        N, T, D = h.shape
        K = int(slot.max().item()) + 1                                              # most chunks of any document in THIS batch (<= MAX_CHUNKS)
        grid = h.new_zeros(B, K, T, D); gm = torch.zeros(B, K, T, dtype=torch.bool, device=h.device)
        grid[owner, slot] = h; gm[owner, slot] = mask.bool()
        H = grid.view(B, K * T, D); M = gm.view(B, K * T)                           # all tokens of a document in one row
        if self.agg == "mean":
            m = M.unsqueeze(-1).float(); doc = (H * m).sum(1) / m.sum(1).clamp_min(1.0)
            return self.head(self.drop(doc)), None
        U = torch.tanh(self.att_w(H))                                               # [B, KT, d_a]
        e = U @ self.att_q.t()                                                      # [B, KT, L] (or [B, KT, 1])
        e = e.masked_fill(~M.unsqueeze(-1), float("-inf"))
        A = e.softmax(dim=1)                                                        # over all tokens of the document
        C = A.transpose(1, 2) @ H                                                   # [B, L, d]  (or [B, 1, d])
        if self.agg == "attn":
            return self.head(self.drop(C[:, 0])), (A if return_attn else None)
        z = (self.drop(C) * self.out_w.unsqueeze(0)).sum(-1) + self.out_b           # z_l = w_l . c_l + b_l
        return z, (A if return_attn else None)


# ---- (c) LoRMA (multiplicative; Bihany, Patel & Modi, ACL Findings 2025) -- the Phase-3/4 code, unchanged ----------------
_PI_IDX = {}
def _pi_index(d, dev):
    """idx[i, j] = (j - i) mod d.  gather(M, 1, idx)[i, j] = M[i, (j - i) mod d]  ==  row i of M rotated right by i."""
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    """h = I(B A) (W0 x).  W0 frozen.  A in R^{r x d_out}, B in R^{d_out x r}, d_out = base.out_features.
       variant 'plus' (LoRMA+):  I(BA) = I_d + (alpha/r) BA ; computed as y + (alpha/r) B(A y), y = W0 x  -> O(b d r), same order as LoRA.
                                 init B = 0  => exactly the Phase-1 layer at step 0.
       variant 'pi'   (LoRMA_pi): I(BA) = rows of BA rotated cyclically (row i by i); init B[:,0]=1, A[0,0]=1, rest of A = 0
                                 => I(BA) = I_d at step 0 (the rotated 'ones x e0' matrix is the identity)."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))
        return y @ Tm.t()


# ---- (d) Bottleneck adapter (Houlsby et al. 2019): serial, after the sub-layer's output projection, before the residual add ----
class AdapterAfter(nn.Module):
    """y = W x (frozen);  out = y + W_up f(W_down y).  W_up = 0 at init  =>  out = y (exactly the Phase-1 sub-layer)."""
    def __init__(self, base: nn.Linear, bottleneck: int):
        super().__init__()
        d = base.out_features
        self.base = base
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5)); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n


def label_prior(train_docs):
    Y = np.stack([d["y"] for d in train_docs]); return np.clip(Y.mean(0), 1e-4, 1 - 1e-4), Y.sum(0)

def build_model(train_docs):
    """Fresh copy of the Phase-1 ENCODER every call (several runs in one session do not contaminate each other)."""
    backbone = copy.deepcopy(lm.encoder).cpu()
    prior, _ = label_prior(train_docs)
    model = DocModelML(backbone, lm_cfg.d_model, L, MAX_CHUNKS, AGG, ATT_DIM, HEAD_DROPOUT, prior if PRIOR_BIAS_INIT else None)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                        # encoder + embeddings + head all trainable
    elif METHOD in ("lorma", "adapter"):
        ref = copy.deepcopy(backbone)
        for p in backbone.parameters(): p.requires_grad = False
        if METHOD == "lorma":
            n = apply_adapter(backbone, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
            print(f"LoRMA[{hp['variant']}] applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        else:
            tg = tuple(t for site, t in (("attn", "o_proj"), ("ffn", "down_proj")) if site in hp["sites"])
            n = apply_adapter(backbone, tg, lambda l: AdapterAfter(l, hp["bottleneck"]))
            n_ad = sum(p.numel() for m_ in backbone.modules() if isinstance(m_, AdapterAfter) for n__, p in m_.named_parameters() if not n__.startswith("base."))
            n_norm = 0
            if hp["train_norms"]:
                for m_ in backbone.modules():
                    if isinstance(m_, (RMSNorm, LayerNorm)):
                        for p in m_.parameters(): p.requires_grad = True; n_norm += p.numel()
            print(f"adapters after {tg}: {n} modules, bottleneck m={hp['bottleneck']} -> {n_ad:,} adapter params + {n_norm:,} norm gains")
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a = backbone.eval()(_ids, attention_mask=_m); b = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        backbone.train(); del ref
    elif METHOD == "partial":
        for p in backbone.parameters(): p.requires_grad = False
        nb = len(backbone.blocks); top = list(range(nb - hp["top_k"], nb))
        for i in top:
            for p in backbone.blocks[i].parameters(): p.requires_grad = True
        for p in backbone.norm_f.parameters(): p.requires_grad = True
        print(f"partial fine-tuning: blocks {top} + final norm unfrozen (lower {nb - hp['top_k']} blocks and the embedding frozen)")
    elif METHOD == "head_only":
        for p in backbone.parameters(): p.requires_grad = False
        print("head-only (feature extraction): the whole Phase-1 backbone is frozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_new = sum(p.numel() for n_, p in model.named_parameters() if not n_.startswith("backbone."))
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%) | of which new (attention + head) {n_new/1e6:.3f}M | "
          f"trainable inside the encoder {(tr - n_new)/1e6:.3f}M")
    model.n_total, model.n_trainable, model.n_head = tot, tr, n_new
    return model


# ---- (d) Losses ---------------------------------------------------------------------------------------------------------
def make_pos_weight(train_docs):
    _, n_pos = label_prior(train_docs); N = len(train_docs)
    if LOSS != "bce" or POS_WEIGHT == "none": return None
    w = np.minimum(np.sqrt((N - n_pos) / np.maximum(n_pos, 1)), MAX_POS_WEIGHT)
    return torch.tensor(w, dtype=torch.float32, device=device)

def multilabel_loss(z, y, pos_weight):
    """z, y: [B, L] (fp32). Returns the mean over documents of the mean over labels."""
    if LOSS == "bce":
        return F.binary_cross_entropy_with_logits(z, y, pos_weight=pos_weight)
    gp, gn, m = ASL["gamma_pos"], ASL["gamma_neg"], ASL["clip"]
    p = torch.sigmoid(z)
    p_m = (p - m).clamp(min=0) if m > 0 else p
    lp = F.logsigmoid(z)                                                            # log p, stable
    ln = torch.log((1 - p_m).clamp(min=1e-8))                                       # log(1 - p_m)
    loss_pos = y * ((1 - p) ** gp if gp > 0 else 1.0) * lp
    loss_neg = (1 - y) * (p_m ** gn if gn > 0 else 1.0) * ln
    return -(loss_pos + loss_neg).mean()

attention: the Phase-1 encoder is bidirectional by construction (key-padding mask only)


In [11]:
# Cell 18 — Inference: documents -> logits [n, L] (+ optional label attention for a subset)
@torch.no_grad()
def predict_logits(model, docs, return_attn_for=None):
    """-> (logits [n, L] float32, seconds, attn dict {doc index: A [tokens of the doc, L] float16} or None)."""
    model.eval()
    _saved = MICRO_TOKENS
    globals()["MICRO_TOKENS"] = 2 * _saved                                          # no activations kept: bigger batches
    try:
        batches = make_microbatches(docs, range(len(docs)), shuffle=False)
    finally:
        globals()["MICRO_TOKENS"] = _saved
    out = np.zeros((len(docs), L), dtype=np.float32); att = {} if return_attn_for is not None else None
    want = set(return_attn_for or [])
    t0 = time.time()
    for idxs in batches:
        ids, mask, owner, slot, _ = build_batch(docs, idxs, with_labels=False)
        ids, mask = torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)
        owner, slot = torch.from_numpy(owner).to(device), torch.from_numpy(slot).to(device)
        need = att is not None and any(i in want for i in idxs)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            z, A = model(ids, mask, owner, slot, len(idxs), return_attn=need)
        out[idxs] = z.float().cpu().numpy()
        if need and A is not None:
            T = ids.shape[1]
            for b, i in enumerate(idxs):
                if i in want:
                    nch = len(docs[i]["chunks"])
                    keep = np.concatenate([np.arange(len(docs[i]["chunks"][j])) + j * T for j in range(nch)])
                    att[i] = A[b, keep].float().cpu().numpy().astype(np.float16)
    if device.type == "cuda": torch.cuda.synchronize()
    return out, time.time() - t0, att

In [12]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, token-budget micro-batches, gradient accumulation, fp16/bf16 autocast)
def param_groups(model):
    hp = HP[METHOD]
    groups = [dict(params=[p for n, p in model.named_parameters() if not n.startswith("backbone.")], lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    bb = [(n, p) for n, p in model.named_parameters() if n.startswith("backbone.") and p.requires_grad]
    if not bb: return groups
    if METHOD == "partial":                                                         # layer-wise LR decay going down
        nb = len(model.backbone.blocks)
        by_lr = defaultdict(lambda: ([], []))
        for n, p in bb:
            mt = re.match(r"backbone\.blocks\.(\d+)\.", n)
            depth = (nb - 1 - int(mt.group(1))) if mt else 0                         # 0 = top block / final norm
            lr = hp["lr"] * hp["decay"] ** depth
            (by_lr[lr][0] if p.ndim >= 2 else by_lr[lr][1]).append(p)
        for lr, (dec, nodec) in sorted(by_lr.items(), reverse=True):
            if dec: groups.append(dict(params=dec, lr=lr, base_lr=lr, weight_decay=hp["wd"]))
            if nodec: groups.append(dict(params=nodec, lr=lr, base_lr=lr, weight_decay=0.0))
        print("partial FT learning rates (top block first):", [f"{lr:.2e}" for lr in sorted(by_lr, reverse=True)])
        return groups
    dec = [p for n, p in bb if p.ndim >= 2 and "tok_emb" not in n]; nodec = [p for n, p in bb if not (p.ndim >= 2 and "tok_emb" not in n)]
    if dec: groups.append(dict(params=dec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    return groups

def quick_dev_score(model, val_docs, Yv):
    z, t_inf, _ = predict_logits(model, val_docs); S = sigmoid(z)
    tg = _best_global(S, Yv)
    return scores(Yv, decide(S, tg))["macro_f1"], scores(Yv, decide(S, 0.5))["macro_f1"], tg, t_inf

def train_doc_model(model, train_docs, val_docs, epochs, max_steps=None):
    pos_w = make_pos_weight(train_docs)
    if pos_w is not None: print(f"pos_weight: min {pos_w.min().item():.2f} median {pos_w.median().item():.2f} max {pos_w.max().item():.2f}")
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    Yv = np.stack([d["y"] for d in val_docs]) if val_docs else None

    n_items = len(train_docs)
    steps_per_epoch = math.ceil(n_items / STEP_DOCS)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} docs/epoch | ~{steps_per_epoch} optimiser steps/epoch ({STEP_DOCS} docs/step) | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=-1.0, epoch=0, state=None)
    t_train = time.time(); n_seen = 0
    for ep in range(1, epochs + 1):
        batches = make_microbatches(train_docs, range(n_items), shuffle=True, seed=SEED + ep)
        model.train(); t0 = time.time(); ep_loss, ep_n, acc_docs = 0.0, 0, 0
        for bi, idxs in enumerate(batches):
            ids, mask, owner, slot, y = build_batch(train_docs, idxs)
            ids, mask, owner, slot, y = (torch.from_numpy(x).to(device) for x in (ids, mask, owner, slot, y))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                z, _ = model(ids, mask, owner, slot, len(idxs))
            loss = multilabel_loss(z.float(), y, pos_w)
            scaler.scale(loss * len(idxs) / STEP_DOCS).backward()                     # docs-weighted, normalised per optimiser step
            lv = loss.item(); losses.append(lv); ep_loss += lv * len(idxs); ep_n += len(idxs); n_seen += len(idxs); acc_docs += len(idxs)
            if acc_docs >= STEP_DOCS or bi == len(batches) - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1; acc_docs = 0
                if step == 20:
                    per = (time.time() - t0) / 20
                    print(f"  >> speed: {per:.2f} s/step | {STEP_DOCS / per:.1f} docs/s | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus dev evaluation)"
                          + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
                if step % 100 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-200:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping (the best epoch so far is kept)"); done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            f_g, f_05, tg, _ = quick_dev_score(model, val_docs, Yv)
            rec.update(val_macro_f1_global=f_g, val_macro_f1_at05=f_05, val_global_thresh=tg)
            if f_g > best["score"]:
                best = dict(score=f_g, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
                torch.save(best, RUN_DIR / "best_trainable.pt")                       # survives a dead session
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, docs_seen=n_seen,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:
        best = dict(score=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [13]:
# Cell 20 — Build data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TRAIN_ALL, min(SMOKE_TRAIN_DOCS, len(TRAIN_ALL)))
    val_docs = _r.sample(DEV_ALL, min(SMOKE_VAL_DOCS, len(DEV_ALL)))
elif FINAL_FIT:
    train_docs = TRAIN_ALL + DEV_ALL; val_docs = []
else:
    train_docs = TRAIN_ALL; val_docs = DEV_ALL
_t0 = time.time()
tokenize_docs(train_docs, "train_smoke" if SMOKE_TEST else ("train_dev" if FINAL_FIT else "train"))
if val_docs: tokenize_docs(val_docs, "dev_smoke" if SMOKE_TEST else "dev")
print(f"tokenisation {time.time() - _t0:.0f}s")
chunk_stats(train_docs, "train")
if val_docs: chunk_stats(val_docs, "dev")

if val_docs:                                        # reference points
    Yv = np.stack([d["y"] for d in val_docs]); pr, _ = label_prior(train_docs)
    _P = np.zeros_like(Yv, dtype=bool); _P[:, np.argsort(-pr)[:4]] = True
    print(f"constant top-4 baseline: dev Macro-F1 {scores(Yv, _P)['macro_f1']:.4f} | TF-IDF + LR (audit): {TFIDF_BASELINE:.3f}")

model = build_model(train_docs)
history, best, stats = train_doc_model(model, train_docs, val_docs, epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | dev Macro-F1 (global threshold, in-sample) {best['score']} | train {stats['train_seconds']/60:.1f} min | "
      f"peak GPU {stats['peak_gpu_gb']:.1f} GB")

tokenise train:   0%|          | 0/42750 [00:00<?, ?it/s]

tokenise dev:   0%|          | 0/10181 [00:00<?, ?it/s]

tokenisation 145s
train: 42750 docs | tokens/doc median 666 p90 4319 | chunks/doc mean 2.18 (1 chunk: 54%, truncated by the view: 12.0%) | share of tokens read 74.7% | docs with a placeholder token in the kept chunks 89%
dev: 10181 docs | tokens/doc median 835 p90 4610 | chunks/doc mean 2.35 (1 chunk: 48%, truncated by the view: 13.9%) | share of tokens read 73.6% | docs with a placeholder token in the kept chunks 90%
constant top-4 baseline: dev Macro-F1 0.0150 | TF-IDF + LR (audit): 0.397
adapters after ('o_proj', 'down_proj'): 12 modules, bottleneck m=64 -> 793,344 adapter params + 6,656 norm gains
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 36.82M | trainable 1.165M (3.16%) | of which new (attention + head) 0.365M | trainable inside the encoder 0.800M
pos_weight: min 1.63 median 7.84 max 12.50
42750 docs/epoch | ~1336 optimiser steps/epoch (32 docs/step) | total 6680 steps
  >> speed: 1.16 s/step | 27.7 docs/s | projected 25.8 min/epoch (129 min for 5 ep

In [14]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    z, _, _ = predict_logits(model, val_docs); S = sigmoid(z)
    tg = _best_global(S, Yv); ev = scores(Yv, decide(S, tg))
    print(f"loss {stats['first_losses']:.4f} -> {stats['last_losses']:.4f} (x{drop:.2f}) | dev Macro-F1 {ev['macro_f1']:.4f} (global thr {tg}) | "
          f"micro-F1 {ev['micro_f1']:.4f} | labels ever predicted {(ev['n_pred'] > 0).sum()}/{L} | statutes/doc predicted {ev['card_pred']:.2f} (gold {ev['card_gold']:.2f})")
    ok = drop < 0.95 and (ev["n_pred"] > 0).sum() >= 20
    print("SMOKE VERDICT:", "OK, the pipeline learns -> run the full job" if ok else
          "NOT OK: loss did not fall clearly or few labels are ever predicted; ~150 steps is short for an adapter - compare METHOD='full' before debugging")
    print("Check the '>> speed' line above: it projects the minutes per epoch of the full run; set EPOCHS / MICRO_TOKENS / TIME_BUDGET_MIN accordingly.")
else:
    print("not a smoke run")

not a smoke run


## Dev analysis (full runs with a dev set)
Everything the report's Phase-5 analysis asks for that can be read from dev: untuned and tuned scores for every threshold strategy, the chosen strategy (held-out), F1 by label frequency and by document length, the most-missed statutes, co-occurring pairs, and whether the label attention looks at the cited passages.

In [15]:
# Cell 22 — Dev analysis
TH = None; th_strategy = None; dev_res = {}; dev_extra = {}
if not SMOKE_TEST and val_docs:
    n_att = min(400, len(val_docs)); att_idx = random.Random(SEED).sample(range(len(val_docs)), n_att) if AGG == "label_attn" else None
    z_val, t_inf, att_val = predict_logits(model, val_docs, return_attn_for=att_idx)
    S_val = sigmoid(z_val)
    dev_extra["infer_ms_per_doc"] = 1000 * t_inf / len(val_docs)
    print(f"inference: {t_inf:.1f}s for {len(val_docs)} docs = {dev_extra['infer_ms_per_doc']:.1f} ms/doc")

    # --- threshold strategies: in-sample and 2-fold held-out ---
    th_strategy, ho = choose_strategy(S_val, Yv)
    print(f"\n{'strategy':10s} {'macro-F1 (in-sample)':>21s} {'held-out':>9s} {'micro-F1':>9s} {'subset acc':>11s} {'stat/doc':>9s}")
    for s in STRATEGIES:
        th = fit_thresholds(S_val, Yv, s); ev = scores(Yv, decide(S_val, th))
        dev_res[s] = dict(macro_f1=ev["macro_f1"], heldout=ho[s], micro_f1=ev["micro_f1"], subset_acc=ev["subset_acc"],
                          macro_p=ev["macro_p"], macro_r=ev["macro_r"], card_pred=ev["card_pred"])
        print(f"{s:10s} {ev['macro_f1']:21.4f} {ho[s]:9.4f} {ev['micro_f1']:9.4f} {ev['subset_acc']:11.4f} {ev['card_pred']:9.2f}" + ("   <- chosen" if s == th_strategy else ""))
    TH = fit_thresholds(S_val, Yv, th_strategy)
    ev = scores(Yv, decide(S_val, TH))
    print(f"\n===== dev | strategy {th_strategy} | Macro-F1 {ev['macro_f1']:.4f} (held-out {ho[th_strategy]:.4f}) | macro-P {ev['macro_p']:.4f} | "
          f"macro-R {ev['macro_r']:.4f} | micro-F1 {ev['micro_f1']:.4f} | subset acc {ev['subset_acc']:.4f}")
    print(f"      vs TF-IDF + LR baseline {TFIDF_BASELINE:.3f}: {ho[th_strategy] - TFIDF_BASELINE:+.4f} (held-out vs held-out)")
    print(f"      statutes/doc: predicted {ev['card_pred']:.2f}, gold {ev['card_gold']:.2f}")
    if th_strategy != "fixed": print(f"      thresholds: min {TH.min():.2f} median {np.median(TH):.2f} max {TH.max():.2f}")
    json.dump(dict(strategy=th_strategy, labels=LABELS, thresholds=[float(t) for t in TH], heldout=ho),
              open(RUN_DIR / "thresholds.json", "w"), indent=1)
    print(f"      saved {RUN_DIR / 'thresholds.json'}  (FINAL_FIT=True: set THRESH_FILE to this file)")

    # --- per-label table, F1 by training frequency ---
    _, n_tr = label_prior(train_docs)
    f1 = ev["f1"]; order = np.argsort(f1)
    dev_extra["per_label"] = {str(LABELS[j]): dict(train_docs=int(n_tr[j]), dev_support=int(ev["support"][j]), P=round(float(ev["p"][j]), 4),
                                                   R=round(float(ev["r"][j]), 4), F1=round(float(f1[j]), 4), threshold=float(TH[j])) for j in range(L)}
    q = np.quantile(n_tr, [0, 1 / 3, 2 / 3, 1]); buckets = []
    print("\nF1 by training frequency (label terciles):")
    for k_ in range(3):
        sel = (n_tr >= q[k_]) & ((n_tr < q[k_ + 1]) if k_ < 2 else (n_tr <= q[k_ + 1]))
        buckets.append(dict(train_docs_lo=int(q[k_]), train_docs_hi=int(q[k_ + 1]), n_labels=int(sel.sum()), mean_f1=float(f1[sel].mean()),
                            mean_p=float(ev["p"][sel].mean()), mean_r=float(ev["r"][sel].mean())))
        print(f"  labels with {int(q[k_]):5d}-{int(q[k_ + 1]):5d} train docs: {sel.sum():3d} labels | F1 {f1[sel].mean():.3f} | P {ev['p'][sel].mean():.3f} | R {ev['r'][sel].mean():.3f}")
    print(f"  Spearman(train frequency, F1) = {np.corrcoef(np.argsort(np.argsort(n_tr)), np.argsort(np.argsort(f1)))[0, 1]:.3f}")
    dev_extra["f1_by_freq"] = buckets
    print("worst 10 statutes (id, train docs, dev support, P, R, F1):",
          [(LABELS[j], int(n_tr[j]), int(ev["support"][j]), round(float(ev["p"][j]), 2), round(float(ev["r"][j]), 2), round(float(f1[j]), 2)) for j in order[:10]])
    print("best 5 statutes:", [(LABELS[j], int(n_tr[j]), round(float(f1[j]), 2)) for j in order[::-1][:5]])

    # --- by document length (token terciles) ---
    P_val = decide(S_val, TH); ntok_v = np.asarray([d["ntok"] for d in val_docs]); cuts = np.percentile(ntok_v, [33.3, 66.7])
    bk = np.digitize(ntok_v, cuts); rows = []
    print("\nby document length (token terciles):  [Macro-F1 over the labels present in the bucket]")
    for b_, name in enumerate(("short", "medium", "long")):
        sel = bk == b_
        e_ = scores(Yv[sel], P_val[sel])
        rows.append(dict(bucket=name, n=int(sel.sum()), median_tokens=float(np.median(ntok_v[sel])), macro_f1=e_["macro_f1"], micro_f1=e_["micro_f1"],
                         truncated_share=float(np.mean([val_docs[i]["nchunks_all"] > MAX_CHUNKS for i in np.where(sel)[0]]))))
        print(f"  {name:6s}: {sel.sum():5d} docs | median {np.median(ntok_v[sel]):6.0f} tokens | Macro-F1 {e_['macro_f1']:.3f} | micro-F1 {e_['micro_f1']:.3f} | "
              f"truncated by the view {100 * rows[-1]['truncated_share']:.0f}%")
    dev_extra["by_length"] = rows

    # --- co-occurring pairs: when both statutes are gold, how often does the model get only one of them? ---
    Ytr_ = np.stack([d["y"] for d in train_docs]).astype(bool); Nt = len(Ytr_); pt = Ytr_.mean(0)
    co = (Ytr_.T.astype(np.int64) @ Ytr_.astype(np.int64)); pairs = []
    for a in range(L):
        for b in range(a + 1, L):
            if co[a, b] >= 50: pairs.append((co[a, b] / Nt / (pt[a] * pt[b]), a, b))
    pairs.sort(reverse=True); pr_rows = []
    print("\nstrongly co-occurring statute pairs (top lift, >= 50 train docs): dev docs with both gold -> both predicted / only one / none")
    Yb = Yv.astype(bool)
    for li, a, b in pairs[:10]:
        both = Yb[:, a] & Yb[:, b]
        if both.sum() == 0: continue
        pa, pb = P_val[both, a], P_val[both, b]
        r_ = dict(pair=[LABELS[a], LABELS[b]], lift=round(float(li), 1), n=int(both.sum()), both=float((pa & pb).mean()), one=float((pa ^ pb).mean()), none=float((~pa & ~pb).mean()))
        pr_rows.append(r_)
        print(f"  {LABELS[a]:3d}+{LABELS[b]:3d} lift {li:5.1f} | n {both.sum():4d} | both {r_['both']:.2f} one {r_['one']:.2f} none {r_['none']:.2f}")
    dev_extra["pairs"] = pr_rows

    # --- does the label attention look at the cited passages? (AGG="label_attn") ---
    if att_val:
        W_NEAR = 16; ratios = []
        for i, A in att_val.items():
            d = val_docs[i]; T_tot = A.shape[0]; near = np.zeros(T_tot, dtype=bool); off = 0
            for c, ph in zip(d["chunks"], d["phpos"]):
                for p in ph: near[max(off, off + p - W_NEAR): off + min(len(c), p + W_NEAR + 1)] = True
                off += len(c)
            if near.sum() == 0 or near.all(): continue
            gold = np.flatnonzero(d["y"] > 0)
            mass = A[:, gold].astype(np.float32).sum(0).mean()                       # attention mass of the gold statutes on any token (=1)
            m_near = A[near][:, gold].astype(np.float32).sum(0).mean()
            ratios.append((m_near / mass, near.mean()))
        if ratios:
            r = np.asarray(ratios)
            dev_extra["attn_evidence"] = dict(n_docs=len(r), mass_near_placeholders=float(r[:, 0].mean()), token_share_near=float(r[:, 1].mean()))
            print(f"\nlabel attention of the GOLD statutes on tokens within +-{W_NEAR} of a placeholder: {100 * r[:, 0].mean():.1f}% of the mass, "
                  f"while those tokens are {100 * r[:, 1].mean():.1f}% of the document ({len(r)} dev docs) -> "
                  f"{r[:, 0].mean() / max(r[:, 1].mean(), 1e-9):.2f}x the uniform share")

    # --- error file ---
    errs = []
    for i in range(len(val_docs)):
        g = set(np.flatnonzero(Yb[i])); p = set(np.flatnonzero(P_val[i]))
        if g != p: errs.append(dict(id=str(val_docs[i]["id"]), gold=sorted(LABELS[j] for j in g), pred=sorted(LABELS[j] for j in p),
                                    missed=sorted(LABELS[j] for j in g - p), extra=sorted(LABELS[j] for j in p - g), ntok=int(val_docs[i]["ntok"])))
    (RUN_DIR / "dev_errors.json").write_text(json.dumps(errs))
    np.save(RUN_DIR / "dev_scores.npy", S_val.astype(np.float16))
    print(f"\n{len(errs)} dev docs with an imperfect statute set -> dev_errors.json | dev_scores.npy saved")
elif FINAL_FIT:
    assert THRESH_FILE is not None, "FINAL_FIT=True needs THRESH_FILE (thresholds.json from the dev run)"
    _tf = json.load(open(THRESH_FILE)); assert _tf["labels"] == LABELS, "label list differs from the dev run"
    TH = np.asarray(_tf["thresholds"]); th_strategy = _tf["strategy"]
    print(f"FINAL_FIT: thresholds from {THRESH_FILE} (strategy {th_strategy})")
else:
    print("no dev analysis (smoke run)")

inference: 127.1s for 10181 docs = 12.5 ms/doc

strategy    macro-F1 (in-sample)  held-out  micro-F1  subset acc  stat/doc
fixed                     0.3747    0.3743    0.4684      0.0281      5.62
global                    0.3805    0.3795    0.4737      0.0391      4.46
per_label                 0.4038    0.3845    0.4919      0.0375      5.11   <- chosen
shrunk                    0.3967    0.3858    0.4903      0.0370      5.09

===== dev | strategy per_label | Macro-F1 0.4038 (held-out 0.3845) | macro-P 0.3915 | macro-R 0.4452 | micro-F1 0.4919 | subset acc 0.0375
      vs TF-IDF + LR baseline 0.397: -0.0125 (held-out vs held-out)
      statutes/doc: predicted 5.11, gold 3.92
      thresholds: min 0.33 median 0.60 max 0.95
      saved /kaggle/working/runs/p5_encdec_adapter/thresholds.json  (FINAL_FIT=True: set THRESH_FILE to this file)

F1 by training frequency (label terciles):
  labels with   272-  543 train docs:  33 labels | F1 0.318 | P 0.325 | R 0.343
  labels with   543- 132

In [16]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(docs, S, th, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    P = decide(S, th)
    sub = [dict(id=d["id"], pred=sorted(int(LABELS[j]) for j in np.flatnonzero(P[i]))) for i, d in enumerate(docs)]   # id keeps its type from test.json
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")          # must sit at the ROOT of the zip
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} docs | statutes/doc {P.sum(1).mean():.2f} | labels ever predicted {(P.sum(0) > 0).sum()}/{L}")
    verify_submission(out_dir / "submission.zip", docs)
    return sub

def verify_submission(zip_path, docs):
    """Re-open the zip like the evaluator would: root file name, one non-empty list of unique valid integer ids per test id, no extras."""
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, pred}"
    want = [str(d["id"]) for d in docs]; got = [str(e["id"]) for e in sub]
    assert len(got) == len(set(got)), "duplicate ids in the submission"
    assert set(got) == set(want), f"id mismatch: missing {len(set(want) - set(got))}, extra {len(set(got) - set(want))}"
    valid = set(LABELS)
    for e in sub:
        assert set(e) == {"id", "pred"}, f"unexpected keys {set(e)}"
        p = e["pred"]
        assert isinstance(p, list) and len(p) > 0, f"{e['id']}: pred must be a non-empty list"
        assert all(isinstance(v, int) and not isinstance(v, bool) for v in p), f"{e['id']}: non-integer statute id"
        assert len(p) == len(set(p)), f"{e['id']}: duplicate statute ids"
        assert set(p) <= valid, f"{e['id']}: unknown statute id(s) {set(p) - valid}"
    print(f"verified: {len(sub)} predictions, unique integer statute ids from the {L}-label set, one per test id")

if not SMOKE_TEST:
    tokenize_docs(TEST_ALL, "test"); chunk_stats(TEST_ALL, "test")
    z_test, t_test, _ = predict_logits(model, TEST_ALL); S_test = sigmoid(z_test)
    np.save(RUN_DIR / "test_scores.npy", S_test.astype(np.float16))
    sub = write_submission(TEST_ALL, S_test, TH, RUN_DIR / "submission")
    print(f"test inference {t_test:.0f}s ({1000 * t_test / len(TEST_ALL):.1f} ms/doc) | thresholds: {th_strategy}")
else:
    print("smoke run: no submission written")

tokenise test:   0%|          | 0/13019 [00:00<?, ?it/s]

test: 13019 docs | tokens/doc median 825 p90 4429 | chunks/doc mean 2.31 (1 chunk: 48%, truncated by the view: 12.9%) | share of tokens read 74.9% | docs with a placeholder token in the kept chunks 90%
wrote /kaggle/working/runs/p5_encdec_adapter/submission/submission.zip | 13019 docs | statutes/doc 5.02 | labels ever predicted 100/100
verified: 13019 predictions, unique integer statute ids from the 100-label set, one per test id
test inference 157s (12.0 ms/doc) | thresholds: per_label


In [17]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full / representation tables)
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, Path): return str(o)
    return o

log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
           att_dim=ATT_DIM, loss=LOSS, pos_weight=POS_WEIGHT, asl=ASL if LOSS == "asl" else None, prior_bias_init=PRIOR_BIAS_INIT,
           final_fit=FINAL_FIT, smoke=SMOKE_TEST, encoder_len=lm_cfg.encoder_len, pos_type=lm_cfg.pos_type,
           hp=HP[METHOD], epochs=EPOCHS, micro_tokens=MICRO_TOKENS, step_docs=STEP_DOCS,
           params_total=model.n_total, params_trainable=model.n_trainable, params_head=model.n_head, history=history, best_epoch=best["epoch"],
           stats=stats, threshold_strategy=th_strategy, dev=dev_res, dev_extra=dev_extra, tfidf_baseline=TFIDF_BASELINE,
           n_train=len(train_docs), n_dev=len(val_docs))
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "agg": AGG, "doc_view": DOC_VIEW, "labels": LABELS,
            "thresholds": None if TH is None else [float(t) for t in TH], "log": _jsonable(log)}, RUN_DIR / f"lsi_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['best_trainable.pt', 'dev_errors.json', 'dev_scores.npy', 'lsi_p5_encdec_adapter.pt', 'run_log.json', 'submission', 'test_scores.npy', 'thresholds.json']


In [18]:
# Cell 25 — Export this run's document-level LOGITS so it can serve as the KD teacher (run #5)
# All Phase-1 models share one tokenizer and the label list is fixed, so these arrays can teach ANY student without this notebook's model code.
# Stored as float16 logits z [L] per document: the student uses sigmoid(z / T) per statute.
if EXPORT_TEACHER and not SMOKE_TEST:
    _t0 = time.time(); _out, _split, _ids = {}, {}, {}
    for _name, _docs in (("train", train_docs), ("dev", val_docs)):
        if not _docs: continue
        _z, _, _ = predict_logits(model, _docs)
        for d, z in zip(_docs, _z):
            _out[d["uid"]] = z.astype(np.float16); _split[d["uid"]] = _name; _ids[d["uid"]] = str(d["id"])
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
                 loss=LOSS, labels=LABELS, params_total=int(model.n_total), params_trainable=int(model.n_trainable), epochs=int(EPOCHS),
                 best_epoch=int(best["epoch"]), final_fit=FINAL_FIT, threshold_strategy=th_strategy,
                 dev_macro_f1=None if not dev_res else float(dev_res[th_strategy]["macro_f1"]),
                 dev_macro_f1_heldout=None if not dev_res else float(dev_res[th_strategy]["heldout"]))
    _path = RUN_DIR / f"teacher_logits_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, docs=_out, split=_split, ids=_ids), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.1f} MB) in {time.time()-_t0:.0f}s | {len(_out)} docs | teacher params {_meta['params_total']/1e6:.1f}M")
    print("caveat: on the TRAIN docs the teacher is overconfident (it was trained on them); its 'dev' logits are out-of-sample.")
else:
    print("teacher export skipped (EXPORT_TEACHER=False or smoke run)")

teacher export skipped (EXPORT_TEACHER=False or smoke run)
